# 04 · Khoảng cách giữa dataset và đường thật (video tự quay trên xe máy)

Input: code + models (notebook 03) + dataset riêng tư `motorbike-frames` gồm:
- `images/*.jpg` (tách bằng `python -m dashcam.frames`, đã làm mờ mặt)
- `labels/*.txt` (nhãn YOLO 52 lớp, gán bằng Label Studio/CVAT, xem docs/quay_video_xe_may.md)
- `frames.csv` (thời điểm, khu vực, độ nét của từng frame)

Câu hỏi: model train trên VNTS (đa phần ảnh từ ô tô / camera cố định) tụt bao nhiêu khi nhìn
từ xe máy, và tụt theo yếu tố nào (đêm, mưa, rung)?

In [ ]:
import glob, json, os, shutil, sys
import numpy as np, pandas as pd

CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE
os.makedirs("models", exist_ok=True)
for f in glob.glob("/kaggle/input/**/models/*", recursive=True):
    shutil.copy(f, "models/")
MB = os.path.dirname(glob.glob("/kaggle/input/**/frames.csv", recursive=True)[0])
!pip install -q onnxruntime opencv-python-headless

In [ ]:
import cv2
from pathlib import Path
from dashcam.evaluate import load_gt
from dashcam.map_eval import map50
from dashcam.pipeline import SignPipeline

meta = pd.read_csv(f"{MB}/frames.csv")
meta["blur"] = pd.qcut(meta["sharpness"], 3, labels=["nhoè", "vừa", "nét"])

def run(mode):
    pipe_preds, gts = {}, {}
    for _, r in meta.iterrows():
        p = Path(MB) / "images" / r["file"]
        img = cv2.imread(str(p))
        res = SignPipeline.load("models", mode=mode).process(img)
        pipe_preds[p.stem] = [(b, float(s) * float(pr.max()), int(pr.argmax()))
                              for b, s, pr in zip(res.boxes, res.det_scores, res.probs) if pr is not None]
        gts[p.stem] = load_gt(p, Path(MB) / "labels", img.shape)
    return pipe_preds, gts

rows = []
for mode in ("one_stage", "two_stage"):
    preds, gts = run(mode)
    rows.append({"mode": mode, "nhóm": "tất cả", "mAP50": map50(preds, gts)[0]})
    for col in ("time", "area", "blur"):
        for val, sub in meta.groupby(col):
            keys = [Path(f).stem for f in sub["file"]]
            m, _ = map50({k: preds[k] for k in keys}, {k: gts[k] for k in keys})
            rows.append({"mode": mode, "nhóm": f"{col}={val}", "mAP50": round(m, 4), "n": len(keys)})
pd.DataFrame(rows)